In [1]:
!pip install -q pymupdf
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q transformers
!pip install -q sentencepiece
!pip install -q gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 34.7 MB/s eta 0:00:00


In [2]:
import fitz
import faiss
import numpy as np
import torch
import gradio as gr

from sentence_transformers import SentenceTransformer
from transformers import pipeline

In [3]:
device = 0 if torch.cuda.is_available() else -1

print("GPU available:", torch.cuda.is_available())

GPU available: False


In [4]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


In [5]:
generator = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    device=device
)

print("Local AI model loaded.")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


...

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'AXK1ForCausalLM', 'AXK2ForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CohereCompassForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM',

Local AI model loaded.


In [6]:
def extract_pdf(pdf_path):

    document = fitz.open(pdf_path)

    pages = []

    for page_number, page in enumerate(document):

        text = page.get_text("text")

        text = " ".join(text.split())

        if text.strip():

            pages.append({
                "page": page_number + 1,
                "text": text
            })

    return pages

In [7]:
def create_chunks(pages, chunk_size=120, overlap=25):

    chunks = []

    for page in pages:

        words = page["text"].split()

        start = 0

        while start < len(words):

            end = start + chunk_size

            chunk_text = " ".join(
                words[start:end]
            )

            if chunk_text.strip():

                chunks.append({
                    "page": page["page"],
                    "text": chunk_text
                })

            start += chunk_size - overlap

    return chunks

In [8]:
def create_vector_database(chunks):

    texts = [
        chunk["text"]
        for chunk in chunks
    ]

    embeddings = embedding_model.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    embeddings = np.asarray(
        embeddings,
        dtype="float32"
    )

    dimension = embeddings.shape[1]

    index = faiss.IndexFlatIP(
        dimension
    )

    index.add(embeddings)

    return index

In [9]:
def retrieve_sections(
    question,
    chunks,
    index,
    top_k=3
):

    query_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype="float32"
    )

    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for score, idx in zip(
        scores[0],
        indices[0]
    ):

        if idx < len(chunks):

            results.append({
                "page": chunks[idx]["page"],
                "text": chunks[idx]["text"],
                "score": float(score)
            })

    return results

In [10]:
def build_prompt(question, results):

    context = ""

    for result in results:

        context += (
            f"\nPage {result['page']}:\n"
            f"{result['text']}\n"
        )

    prompt = f"""
You are a financial document assistant.

Use ONLY the annual report information below.

Question:
{question}

Annual Report Information:
{context}

Instructions:

- Answer only from the provided annual report.
- Do not invent numbers.
- Explain the answer in simple English.
- Keep financial figures exactly as written.
- Mention relevant page numbers.
- If the information is unavailable, say:
  "I could not find this information in the annual report."

Give the response like this:

Answer:
Simple Explanation:
Source Pages:
"""

    return prompt

In [11]:
def answer_question(
    question,
    chunks,
    index
):

    results = retrieve_sections(
        question,
        chunks,
        index,
        top_k=3
    )

    prompt = build_prompt(
        question,
        results
    )

    output = generator(
        prompt,
        max_new_tokens=200,
        do_sample=False,
        truncation=True
    )

    answer = output[0]["generated_text"]

    return answer, results

In [14]:
from google.colab import files

uploaded = files.upload()

Saving AR-25-26-20260719_Accesible-20260903.pdf to AR-25-26-20260719_Accesible-20260903.pdf


In [15]:
pdf_path = list(uploaded.keys())[0]
pages = extract_pdf(pdf_path)

print(
    "Pages extracted:",
    len(pages)
)

Pages extracted: 131


In [16]:
chunks = create_chunks(pages)

print(
    "Chunks created:",
    len(chunks)
)

Chunks created: 451


In [17]:
index = create_vector_database(
    chunks
)

print(
    "FAISS database created successfully."
)

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

FAISS database created successfully.


In [19]:
question = """
What was the company's revenue?
"""

answer, sources = answer_question(
    question,
    chunks,
    index
)

print(answer)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a financial document assistant.

Use ONLY the annual report information below.

Question:

What was the company's revenue?


Annual Report Information:

Page 34:
Chart 3.1 Economic Activity/Sector-wise distribution of Active Companies as on December 31, 2025 TS&C Agri & Allied 4.01% 4.49% r Trading 13.64% '-. RE&R 4.96% Others 0.15% M&Q 0.77% Manufacturing 19.08% Insurance Finance EG&WS 0.07% 3-86% 1.60% Business Services 25.56% CP&S 14.29% -........_ Construction 7.52% [*CP&S- Community, Personal and Social Services, *Agri & Allied -Agriculture and Allied Activities, *RE&R - Real Estate and Renting, *TS&C -Transport, Storage and Communications, *EG&WS- Electricity, Gas and Water Supply Companies, *M&Q- Mining and Quarrying] 3.3.2 Duringthe monthof December 01, 2024 to December 31, 2024, a total of 12,584 companies were registered with a collective paid-up capital of~ 692.68 crore. Of these, 7 were Government Companies with paid-up capital of~ 10.59 crore and 12,577 were

Page

In [20]:
for result in sources:

    print("=" * 70)

    print(
        "Page:",
        result["page"]
    )

    print(
        "Similarity:",
        round(result["score"], 3)
    )

    print(
        result["text"]
    )

Page: 34
Similarity: 0.448
Chart 3.1 Economic Activity/Sector-wise distribution of Active Companies as on December 31, 2025 TS&C Agri & Allied 4.01% 4.49% r Trading 13.64% '-. RE&R 4.96% Others 0.15% M&Q 0.77% Manufacturing 19.08% Insurance Finance EG&WS 0.07% 3-86% 1.60% Business Services 25.56% CP&S 14.29% -........_ Construction 7.52% [*CP&S- Community, Personal and Social Services, *Agri & Allied -Agriculture and Allied Activities, *RE&R - Real Estate and Renting, *TS&C -Transport, Storage and Communications, *EG&WS- Electricity, Gas and Water Supply Companies, *M&Q- Mining and Quarrying] 3.3.2 Duringthe monthof December 01, 2024 to December 31, 2024, a total of 12,584 companies were registered with a collective paid-up capital of~ 692.68 crore. Of these, 7 were Government Companies with paid-up capital of~ 10.59 crore and 12,577 were
Page: 33
Similarity: 0.43
SI. Economic Activity No Private Paid Up Number Capital Public Paid Up Number Capital Total Paid Up Number Capital iv Food 

In [21]:
questions = [

    "What was the total revenue?",

    "What was the net profit?",

    "What were the main business risks?",

    "How much debt does the company have?",

    "Explain the company's cash flow.",

    "What are the company's major business segments?"

]

In [22]:
for question in questions:

    print("\n")
    print("=" * 80)

    print(
        "QUESTION:",
        question
    )

    answer, sources = answer_question(
        question,
        chunks,
        index
    )

    print(answer)



QUESTION: What was the total revenue?


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a financial document assistant.

Use ONLY the annual report information below.

Question:
What was the total revenue?

Annual Report Information:

Page 34:
Chart 3.1 Economic Activity/Sector-wise distribution of Active Companies as on December 31, 2025 TS&C Agri & Allied 4.01% 4.49% r Trading 13.64% '-. RE&R 4.96% Others 0.15% M&Q 0.77% Manufacturing 19.08% Insurance Finance EG&WS 0.07% 3-86% 1.60% Business Services 25.56% CP&S 14.29% -........_ Construction 7.52% [*CP&S- Community, Personal and Social Services, *Agri & Allied -Agriculture and Allied Activities, *RE&R - Real Estate and Renting, *TS&C -Transport, Storage and Communications, *EG&WS- Electricity, Gas and Water Supply Companies, *M&Q- Mining and Quarrying] 3.3.2 Duringthe monthof December 01, 2024 to December 31, 2024, a total of 12,584 companies were registered with a collective paid-up capital of~ 692.68 crore. Of these, 7 were Government Companies with paid-up capital of~ 10.59 crore and 12,577 were

Page 33:
S

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a financial document assistant.

Use ONLY the annual report information below.

Question:
What was the net profit?

Annual Report Information:

Page 33:
SI. Economic Activity No Private Paid Up Number Capital Public Paid Up Number Capital Total Paid Up Number Capital iv Food stuffs 59,774 67,009.33 2,540 53,623.79 62,314 1,20,633.12 V Paper & Paper products, Publishing, printing, and reproduction ofrecorded media 19,714 16,968.47 808 11,748.78 20,522 28,717.26 vi Others 25,780 23,197.23 630 18,530.42 26,410 41,727.64 vii Leather & products therec 4,220 5,640.71 191 1,635.96 4,411 7,276.67 viii Wood Products 3,983 2,960.18 181 1,452.52 4,164 4,412.69 2. Construction 1,47,180 1,94,954.27 4,382 2,69,313.73 1,51,562 4,64,268.01 3. Electricity, Gas & Water companies 30,114 2,03,098.83 2,121 9,82,318.61 32,235 11,85,417.44 4. Mining & Quarrying 14,810 38,428.54 721 64,203.12 15,531 1,02,631.66 III Services 12,91,591 13,84,404.34 45,947 17,02,381 13,37,536 30,86,785.34 1. Business Se

...


You are a financial document assistant.

Use ONLY the annual report information below.

Question:
How much debt does the company have?

Annual Report Information:

Page 57:
high levels by the end of 2015. By September, 2016, it reached about 9 percent of gross loans of all banks and 12 percent of gross loans of public sector banks, which accounted for more than 80 percent of total NPAs. On the corporate side, major companies were operating with interest coverage ratio of less than 1, implying inability to service debt obligations. Thus, what emerged is popularly referred to as the 'Twin Balance Sheet' problem where the banks were reeling under the stress of bad loans and corporates were unable to service their debt. 6.3.2 With time, it was deemed expedient to make the resolution of unviable entities effective, easily accessible and provide for ease of exit for firms.

Page 56:
hampered confidence of lenders and consequently debt market. While secured credit from banks was predominant 

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a financial document assistant.

Use ONLY the annual report information below.

Question:
Explain the company's cash flow.

Annual Report Information:

Page 19:
of all books & papers and assets and properties of company under liquidation; assessing the financial position of company on the basis of the Statement of Affairs filed with him; filing of claims against debtors for realization of debts due to the company; sale of movable and immovable assets of the company taken into possession by OL; invitation of claims from creditors/workers; adjudication of claims and settlement of list of creditors; payment to creditors/workers by way of dividend; payment of subsequent interest to creditors/workers in the event of there being a surplus after payment in full of all the claims admitted to proof, settlement of list of contributories (i.e. persons liable to contribute towards the assets of the company in

Page 34:
Chart 3.1 Economic Activity/Sector-wise distribution of Active Compani

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a financial document assistant.

Use ONLY the annual report information below.

Question:
What are the company's major business segments?

Annual Report Information:

Page 19:
provisions of the Companies Act are dealt with by various Divisions /Sections / Cells of the Ministry and various organisations under the administrative --------------------< 10 >------------------­

Page 19:
wholly­ owned subsidiary company; (iii) two or more start-up companies or (iv) one or more start­ up companies with one or more small company. Organisational set-up at Headquarters 2.3.1 The Headquarters of MCA is organized into various Divisions/ Sections/ Cells for administering and regulating various provisions of the Companies Act and other Acts administered by the Ministry. Matters relating to working and administration of Companies Act are discussed in Chapter-III, while the matters relating to the Limited Liability Partnership Act and the Competition Act are dealt with in Chapters IV and V, r

In [23]:
app_chunks = None
app_index = None

In [24]:
def process_report(pdf_file):

    global app_chunks
    global app_index

    if pdf_file is None:
        return "Please upload an annual report."

    pages = extract_pdf(
        pdf_file
    )

    app_chunks = create_chunks(
        pages
    )

    app_index = create_vector_database(
        app_chunks
    )

    return f"""
Annual report processed successfully.

Pages processed: {len(pages)}

Searchable sections: {len(app_chunks)}

You can now ask questions about the report.
"""

In [25]:
def ask_report(question):

    if app_chunks is None:
        return "Please upload an annual report first."

    if not question.strip():
        return "Please enter a question."

    answer, sources = answer_question(
        question,
        app_chunks,
        app_index
    )

    source_pages = sorted(
        set(
            result["page"]
            for result in sources
        )
    )

    source_text = ", ".join(
        str(page)
        for page in source_pages
    )

    return (
        answer +
        "\n\nRetrieved PDF pages: " +
        source_text
    )

In [26]:
with gr.Blocks() as demo:

    gr.Markdown(
        """
        # 📊 AI Financial Document Analyser



        ## AI Financial & Business Document Analyser

        Upload a company annual report and ask questions
        about its finances, risks and business performance.

        The system finds relevant sections of the report
        and explains them in simple language.
        """
    )

    pdf_file = gr.File(
        label="Upload Annual Report",
        file_types=[".pdf"],
        type="filepath"
    )

    process_button = gr.Button(
        "Process Report"
    )

    status = gr.Textbox(
        label="Document Status"
    )

    process_button.click(
        process_report,
        pdf_file,
        status
    )

    question = gr.Textbox(
        label="Ask a Question",
        placeholder=
        "Example: What was the company's revenue?"
    )

    ask_button = gr.Button(
        "Analyse"
    )

    answer_box = gr.Textbox(
        label="AI Analysis",
        lines=12
    )

    ask_button.click(
        ask_report,
        question,
        answer_box
    )


demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://677949710c5498b341.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
